# Step 1 : Import dependencies

In [ ]:
import os
import gymnasium
from stable_baselines3 import PPO 
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy

# Step 2 : Load Environment

In [ ]:
env_name = "CartPole-v1"
env = gymnasium.make(env_name)

episodes = 5
for episode in range(1,episodes+1):
    state,info = env.reset()
    done = False 
    score = 0

    while not done:
        action = env.action_space.sample()
        n_state, reward,terminated, truncated, info = env.step(action)
        done = terminated 
        score += reward
    print(f"Episode{episode} ,Score{score}, Info {info}, Reward : {reward}")

    


## Understanding the environment

In [ ]:
env.action_space

In [ ]:
env.observation_space

# Step 3 - Training a model

In [ ]:
# Making the directories
log_path = os.path.join('Training',"Logs")
PPO_Path = os.path.join('Training/Saved Models','PPO')

In [ ]:
env = gymnasium.make(env_name,render_mode="human")
env = DummyVecEnv([lambda:env])
model = PPO('MlpPolicy', env, verbose=1, tensorboard_log=log_path)

In [ ]:
model.learn(total_timesteps=20000)

In [ ]:

model.save(PPO_Path)

In [ ]:
del model

In [ ]:

model = PPO.load(PPO_Path,env=env)

# 4.Evaluation

In [ ]:
evaluate_policy(model, env, n_eval_episodes=5,render = False)

# 5 Testing

In [ ]:
episodes = 5
env = gymnasium.make(env_name,render_mode = "human")
for episode in range(1,episodes+1):
    obs,info = env.reset()
    done = False 
    score = 0

    while not done:
        env.render()
        action,_ = model.predict(obs)
        obs, reward,terminated, truncated, info = env.step(action)
        done = terminated   
        score += reward
    print(f"Episode{episode} ,Score{score}, Info {info}, Reward : {reward}")

env.close()


# 6. Viewing logs in tensorboard

In [ ]:
log_path

In [ ]:
!tensorboard --logdir={log_path}

In [ ]:
import mujoco.viewer


mujoco.viewer.launch()

# 7. Adding callback to the training stage

In [ ]:
from stable_baselines3.common.callbacks import EvalCallback, StopTrainingOnRewardThreshold

In [ ]:
save_path = os.path.join("Training","Saved Models")

In [ ]:
stop_callback = StopTrainingOnRewardThreshold(reward_threshold=200,verbose=1)
eval_callback = EvalCallback(env,
                             callback_on_new_best=stop_callback,
                             eval_freq=10000,
                             best_model_save_path=save_path,
                             verbose=1)

In [ ]:
model = PPO("MlpPolicy",env,verbose=1,tensorboard_log=log_path)

In [ ]:
model.learn(total_timesteps=20000,callback=eval_callback)

# 8. Changing policies

In [ ]:
net_arch = [dict(pi = [128,128,128,128],vf = [128,128,128,128])]

In [ ]:
model = PPO('MlpPolicy', env, verbose=1, tensorboard_log=log_path,policy_kwargs={"net_arch":net_arch})

In [ ]:
model.learn(total_timesteps=20000,callback=eval_callback)

# 9. Using alternate algorithm

In [ ]:
from stable_baselines3 import DQN

In [ ]:
env = gymnasium.make(env_name)
model = DQN('MlpPolicy', env, verbose=1, tensorboard_log=log_path)

In [ ]:
model.learn(total_timesteps=200000)

In [ ]:
env.close()

## test for alternate algorithm

In [ ]:
episodes = 5
env = gymnasium.make(env_name,render_mode = "human")
for episode in range(1,episodes+1):
    obs,info = env.reset()
    done = False 
    score = 0

    while not done:
        env.render()
        action,_ = model.predict(obs)
        obs, reward,terminated, truncated, info = env.step(action)
        done = terminated   
        score += reward
    print(f"Episode{episode} ,Score{score}, Info {info}, Reward : {reward}")

env.close()

dqn is not that effective